In [0]:
dbutils.secrets.list('storage_key')

In [0]:
storage_account = "azurestoragepramod"
storage_key = dbutils.secrets.get(
    scope="storage_key",
    key="storage-key"
)
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

In [0]:
dbutils.widgets.text("file",'')
dbutils.widgets.text("pipeline_run_id",'')
dbutils.widgets.text("pipline_name",'')
dbutils.widgets.text("source_name",'')
dbutils.widgets.text("file_name",'')
dbutils.widgets.text("landing_area_path",'')
dbutils.widgets.text("pipline_status", "")
dbutils.widgets.text("pipline_start_time", "")
dbutils.widgets.text("pipline_end_time", "")
dbutils.widgets.text("pipeline_error_message", "")

file = dbutils.widgets.get("file")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
pipline_name = dbutils.widgets.get("pipline_name")
source_name = dbutils.widgets.get("source_name")
file_name = dbutils.widgets.get("file_name")
landing_area_path = dbutils.widgets.get("landing_area_path")
pipline_status = dbutils.widgets.get("pipline_status")
pipline_start_time = dbutils.widgets.get("pipline_start_time")
pipline_end_time = dbutils.widgets.get("pipline_end_time")
pipeline_error_message = dbutils.widgets.get("pipeline_error_message")

In [0]:
%sql
create schema if not exists bronze;

In [0]:
from pyspark.sql.types import *

if source_name == "employer":
    expected_schema = StructType([
        StructField("employer_id", StringType(), True),
        StructField("employer_name", StringType(),False),
        StructField("payroll_provider", StringType(),False),
        StructField("status", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "benificiary":
    expected_schema = StructType([
        StructField("beneficiary_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("beneficiary_name", StringType(),False),
        StructField("relationship", StringType(),False),
        StructField("allocation_pct", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "employement_history":
    expected_schema = StructType([
        StructField("history_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("old_title", StringType(),False),
        StructField("new_title", StringType(),False),
        StructField("effective_date", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "participants":
    expected_schema = StructType([
        StructField("participant_id", StringType(), True),
        StructField("employee_number", StringType(),False),
        StructField("first_name", StringType(),False),
        StructField("last_name", StringType(),False),
        StructField("employer_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("email", StringType(),False),
        StructField("mobile", StringType(),False),
        StructField("status", StringType(),False),
        StructField("department", StringType(),False),
        StructField("last_modified_date", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "payroll_calender":
    expected_schema = StructType([
        StructField("payroll_run_id", StringType(), True),
        StructField("period", StringType(),False),
        StructField("frequency", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "plan_enrollment":
    expected_schema = StructType([
        StructField("enrollment_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("contribution_pct", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "plan_master":
    expected_schema = StructType([
        StructField("plan_id", StringType(), True),
        StructField("plan_name", StringType(),False),
        StructField("plan_type", StringType(),False),
        StructField("employer_id", StringType(),False),
        StructField("match_pct", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])
if source_name == "eligibility":
    expected_schema = StructType([
        StructField("eligibility_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("status", StringType(),False),
        StructField("_rescued_data", StringType(),False)
    ])


In [0]:
landing_path=f'abfss://landing-area@azurestoragepramod.dfs.core.windows.net/{employeer}/'
checkpoint_path='abfss://checkpoint@azurestoragepramod.dfs.core.windows.net/employeer/'
schema_path='abfss://schema@azurestoragepramod.dfs.core.windows.net/employeer/'

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header","true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(landing_path)
)

In [0]:
from pyspark.sql.types import *
expected = [f.name for f in expected_schema]
actual = [f.name for f in df.schema]

print(expected)
print(actual)

missing = set(expected) - set(actual)
# print(len(missing))
extra = set(actual) - set(expected)
# print(len(extra))

datatype_mismatch = []
print(type(len(missing)+len(extra)))
if len(missing)+len(extra) >0:
    if len(missing)>0:
        datatype_mismatch.append(list(missing))
    if len(extra)>0:
        datatype_mismatch.append(list(extra))

source_count=spark.read.option("header", "true").csv(landing_path + "/EMPLOYER_MASTER.csv").count()
bronze_count = spark.read.table("pramoddatabricks.bronze.employer").count()
print(source_count)
print(bronze_count)

print(datatype_mismatch)




In [0]:
(
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(once=True)
    .toTable("pramoddatabricks.bronze.employer")
)

In [0]:
%sql
select * from pramoddatabricks.bronze.employer